In [3]:
import pandas as pd

In [4]:
# Start Spark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count,avg, min, max

In [5]:
# Load Data

spark = SparkSession.builder.appName("CSV Loader").getOrCreate()
df = spark.read.csv(r"C:\Users\khand\Downloads\Sample - Superstore.csv",header=True,inferSchema=True)

In [6]:
# show first few row
df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [8]:
# columns name
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

In [9]:
df.dtypes

[('Row ID', 'int'),
 ('Order ID', 'string'),
 ('Order Date', 'string'),
 ('Ship Date', 'string'),
 ('Ship Mode', 'string'),
 ('Customer ID', 'string'),
 ('Customer Name', 'string'),
 ('Segment', 'string'),
 ('Country', 'string'),
 ('City', 'string'),
 ('State', 'string'),
 ('Postal Code', 'int'),
 ('Region', 'string'),
 ('Product ID', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('Sales', 'string'),
 ('Quantity', 'string'),
 ('Discount', 'string'),
 ('Profit', 'double')]

In [10]:
df.printSchema()


root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [11]:
# Data Cleaning
df = df.dropDuplicates()

In [12]:
df.count()-df.distinct().count()

0

In [13]:
# check missing values
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [14]:
# Step 5: Filter Data

df.filter(df["Region"] == "East").show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------------+-----------+------+---------------+---------------+------------+--------------------+--------------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|               State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|         Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------------+-----------+------+---------------+---------------+------------+--------------------+--------------+--------+--------+--------+
|   303|CA-2016-142545|10/28/2016| 11/3/2016|Standard Class|   JD-15895| Jonathan Doherty|  Corporate|United States|   Belleville|          New Jersey|       7109|

In [15]:
df.filter(df["Category"] == "Furniture").show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|         City|               State|Postal Code| Region|     Product ID| Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+-------------+--------------------+-----------+-------+---------------+---------+------------+--------------------+--------+--------+--------+---------+
|   717|CA-2014-130092| 1/11/2014| 1/14/2014|   First Class|   SV-20365|       Seth Vernon|   Consumer|United States|        Dover|            Delaware|      19901|   East|FUR-FU-10000010|Fu

In [16]:
df.filter(df["Segment"] == "Consumer").show()

+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name| Segment|      Country|         City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|         Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+--------------+--------+--------+---------+
|   392|US-2014-135972| 9/21/2014| 9/23/2014|  Second Class|   JG-15115|        Jack Garza|Consumer|United States|   Des Moines|  Washington|      98198|   West|TEC-PH-10003012|     Techn

In [17]:
df.filter(df["Sales"] > 500).show()

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|               State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+----------+
|  1828|CA-2016-109344|  2/8/2016| 2/11/2016|  Second Class|   CH-12070|      Cathy Hwang|Home Office|United States|      Raleigh|      North Carolina|      27604|  South|T

In [27]:
# Transform Data
df = df.withColumn("Sales", col("Sales").cast("float"))
df = df.withColumn("Discount", col("Discount").cast("float"))
df = df.withColumn("Quantity",col("Quantity").cast("float"))

In [20]:
df = df.withColumnRenamed("Sales", "TotalSales")
df = df.withColumnRenamed("Ship Mode", "Shipment Mode")
df = df.withColumnRenamed("Ship Date", "Shipment Date")

In [21]:
df.dtypes


[('Row ID', 'int'),
 ('Order ID', 'string'),
 ('Order Date', 'string'),
 ('Shipment Date', 'string'),
 ('Shipment Mode', 'string'),
 ('Customer ID', 'string'),
 ('Customer Name', 'string'),
 ('Segment', 'string'),
 ('Country', 'string'),
 ('City', 'string'),
 ('State', 'string'),
 ('Postal Code', 'int'),
 ('Region', 'string'),
 ('Product ID', 'string'),
 ('Category', 'string'),
 ('Sub-Category', 'string'),
 ('Product Name', 'string'),
 ('TotalSales', 'float'),
 ('Quantity', 'string'),
 ('Discount', 'float'),
 ('Profit', 'double')]

In [22]:
# Step 7: Aggregation
# (total rows)
print("Total Rows:", df.count())

Total Rows: 9994


In [23]:
# (minimum and maximum values)
df.select(
    min("Quantity").alias("Min Quantity"),
    max("Quantity").alias("Max Quantity"),
    min("Discount").alias("Min Discount"),
    max("Discount").alias("Max Discount"),
    min("Profit").alias("Min Profit"),
    max("Profit").alias("Max Profit")
).show()


+-------------+------------+------------+------------+----------+----------+
| Min Quantity|Max Quantity|Min Discount|Max Discount|Min Profit|Max Profit|
+-------------+------------+------------+------------+----------+----------+
| 1040 sheets"|      98.352|         0.0|     295.056| -6599.978|  8399.976|
+-------------+------------+------------+------------+----------+----------+



In [37]:
df.groupBy("Category").agg(
    sum("TotalSales").alias("Total Sales")
).show()

+---------------+-----------------+
|       Category|      Total Sales|
+---------------+-----------------+
|Office Supplies|703502.9273704886|
|      Furniture|733046.8596462011|
|     Technology|835900.0648635626|
+---------------+-----------------+



In [38]:
df.groupBy("Category").avg("Profit").show()

+---------------+-----------------+
|       Category|      avg(Profit)|
+---------------+-----------------+
|Office Supplies|20.01873189512115|
|      Furniture|9.281672418670434|
|     Technology| 78.7159158635625|
+---------------+-----------------+



In [39]:
df.groupBy("Region").agg(avg("Profit").alias("Average Profit")).show()

+-------+------------------+
| Region|    Average Profit|
+-------+------------------+
|  South|28.796506790123438|
|Central| 17.28390142057684|
|   East| 32.16399462780901|
|   West|33.500999531689054|
+-------+------------------+



In [ ]:
df.toPandas().to_csv(
    r"C:\Users\HP\Downloads\results.csv",
    index=False
)